In [135]:
import numpy as np
import pandas as pd
import os
import re
from datetime import datetime
from difflib import get_close_matches
import string
import time
import pycountry

**Importing the file**

In [136]:
shark_attack_original = pd.read_excel("GSAF5.xls")
shark_attack_original.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,24th May,2026.0,Unprovoked,Australia,Queensland,Kennedy Shoal,Spearfishing,Michael Jensz,M,39,...,Undetermined Bull shark most likely,Simon de Marchi: 9 News: 7 News: ABC News:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17th May,2026.0,Questionable,USA,Maryland,Assateague State Park,Surfing,Brendan Oster,M,?,...,Blue fish bite most probable,Keithe Cowley: Delmarva Now:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16th May,2026.0,Unprovoked,Australia,Western Australia,Horseshoe Reef Rottnest Island,Skindiving,Steven Mattaboni,M,38,...,Great White Shark,ABC News: 9 News: 7 News: Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,14th April,2026.0,UNprovoked,Maldives,Gaafu Alif Atoll,Kooddoo,Swimming,Not stated - on honeymoon,M,?,...,Unknown,The U.S. Sun: Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3rd April,2026.0,Unprovoked,Australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,Surfing,Oliver Tokic-Bensley,M,16,...,Bronze Whaler,ABC News: The Guardian: Andrew Currie and Bob...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [137]:
shark_attack_original. info()

<class 'pandas.DataFrame'>
RangeIndex: 7090 entries, 0 to 7089
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7090 non-null   object 
 1   Year            7088 non-null   float64
 2   Type            7072 non-null   str    
 3   Country         7040 non-null   str    
 4   State           6603 non-null   str    
 5   Location        6523 non-null   str    
 6   Activity        6507 non-null   str    
 7   Name            6872 non-null   str    
 8   Sex             6512 non-null   str    
 9   Age             4096 non-null   object 
 10  Injury          7054 non-null   str    
 11  Fatal Y/N       6529 non-null   object 
 12  Time            3563 non-null   object 
 13  Species         3959 non-null   str    
 14  Source          7070 non-null   object 
 15  pdf             6799 non-null   object 
 16  href formula    6794 non-null   str    
 17  href            6796 non-null   str    
 18 

In [138]:
#Droping columns
shark_attack_df = shark_attack_original.copy()
shark_attack_df = shark_attack_df.drop(columns=[
    'Source', 'pdf', 'href formula', 'href',
    'Case Number', 'Case Number.1',
    'original order', 'Unnamed: 21', 'Unnamed: 22', 'Name', 'State',
    'Location', 'Time', 'Age'
    ])
shark_attack_df.head()
shark_attack_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7090 entries, 0 to 7089
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       7090 non-null   object 
 1   Year       7088 non-null   float64
 2   Type       7072 non-null   str    
 3   Country    7040 non-null   str    
 4   Activity   6507 non-null   str    
 5   Sex        6512 non-null   str    
 6   Injury     7054 non-null   str    
 7   Fatal Y/N  6529 non-null   object 
 8   Species    3959 non-null   str    
dtypes: float64(1), object(2), str(6)
memory usage: 1.0+ MB


**YEAR AND DATE COLUMN CLEANING**

In [139]:
shark_attack_df["Year"].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [140]:
#Filtering before 1960
shark_attack_df = shark_attack_df[
    shark_attack_df["Year"] >= 1960
]

shark_attack_df.head()

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species
0,24th May,2026.0,Unprovoked,Australia,Spearfishing,M,Bite wounds to the head,Y,Undetermined Bull shark most likely
1,17th May,2026.0,Questionable,USA,Surfing,M,Bite wound to the hand,N,Blue fish bite most probable
2,16th May,2026.0,Unprovoked,Australia,Skindiving,M,Severe injuries both legs,Y,Great White Shark
3,14th April,2026.0,UNprovoked,Maldives,Swimming,M,Leg strpped off flesh later amputated in hospital,N,Unknown
4,3rd April,2026.0,Unprovoked,Australia,Surfing,M,Bite wound to R ankle,N,Bronze Whaler


In [141]:
shark_attack_df["Year"].info()

<class 'pandas.Series'>
Index: 4804 entries, 0 to 4804
Series name: Year
Non-Null Count  Dtype  
--------------  -----  
4804 non-null   float64
dtypes: float64(1)
memory usage: 75.1 KB


In [142]:
shark_attack_df["Year"].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017., 2016., 2015., 2014., 2013., 2012., 2011., 2010., 2009.,
       2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001., 2000.,
       1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993., 1992.,
       1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985., 1983.,
       1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975., 1974.,
       1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965., 1964.,
       1963., 1962., 1961., 1960.])

In [143]:
shark_attack_df["Year"] = shark_attack_df["Year"].astype(int)

In [144]:
shark_attack_df["Year"].unique()

array([2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016,
       2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005,
       2004, 2003, 2002, 2001, 2000, 1999, 1998, 1997, 1996, 1995, 1984,
       1994, 1993, 1992, 1991, 1990, 1989, 1969, 1988, 1987, 1986, 1985,
       1983, 1982, 1981, 1980, 1979, 1978, 1977, 1976, 1975, 1974, 1973,
       1972, 1971, 1970, 1968, 1967, 1966, 1965, 1964, 1963, 1962, 1961,
       1960])

In [145]:
#Starting Date cleaning
shark_attack_df["Date_parsed"] = pd.to_datetime(
    shark_attack_df["Date"],
    errors="coerce"
)
shark_attack_df["Date_parsed"].sample(20)

/tmp/ipykernel_10728/1720687397.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  shark_attack_df["Date_parsed"] = pd.to_datetime(


68     2025-05-26 00:00:00.000000000
2878   1999-06-19 00:00:00.000000000
2311   2005-09-04 00:00:00.000000000
4115   1970-01-01 00:00:00.000001972
630    2019-07-04 00:00:00.000000000
3739   1982-07-19 00:00:00.000000000
3885   1978-07-27 00:00:00.000000000
3910   1977-08-05 00:00:00.000000000
3194   1994-05-31 00:00:00.000000000
1278   2014-08-06 00:00:00.000000000
3667   1984-02-11 00:00:00.000000000
2988   1997-08-24 00:00:00.000000000
2799   2000-07-04 00:00:00.000000000
1213   2015-01-23 00:00:00.000000000
2560   2003-01-02 00:00:00.000000000
3411   1989-12-19 00:00:00.000000000
1212   2015-01-24 00:00:00.000000000
4063                             NaT
1638   2011-08-18 00:00:00.000000000
4196   1969-08-01 00:00:00.000000000
Name: Date_parsed, dtype: datetime64[ns]

In [146]:
#First clean Up of dates with no format
invalid_dates = shark_attack_df.loc[
    shark_attack_df["Date_parsed"].isna(),
    "Date"
]

invalid_dates.unique()

array(['24th May', '17th May', '16th May', '14th April', '3rd April',
       '26th March', '25th March', '18th March', '14th March',
       '10th March', '5th March', '22nd February', '6th February',
       '29th January', '24th January', '20th January', '19th January',
       '18th January', '13th January', '10th January', '8th January',
       '5th January', '3rd January ', '25th December', '21st December',
       '12th December', '9th December', '27th November ', '10th November',
       '9th November', '5th November', '4th November', '14th October',
       '11th October', '7th October', '29th September', '27th September',
       '6th September', '1st September', '30th August', '18th August',
       '17th August', '16th August', '7th August', '1st August',
       '28th July', '25th July', '22nd July', '20th July', '19th July',
       '18th July', '15th July', '6th July ', '6th July', '4th July',
       '29th June', '25th June', '22nd June', '17th June', '31st May',
       'Reported 0

In [147]:
invalid_dates.value_counts().sum()

np.int64(307)

In [148]:
#Clean 2 Word replacement
shark_attack_df["Date"] = (
    shark_attack_df["Date"]
    .str.replace("Reported ", "", regex=False)
    .str.replace("Late ", "", regex=False)
    .str.replace("Early ", "", regex=False)
    .str.replace("Mid ", "", regex=False)
    .str.replace("Summer ", "", regex=False)
    .str.replace("Winter ", "", regex=False)
    .str.replace("Fall ", "", regex=False)
    .str.replace("Ca.", "", regex=False)
    .str.replace("Circa", "", regex=False)
    .str.replace("Summer-2008", "", regex=False)
    .str.replace("Circa", "", regex=False)
    .str.replace("Last incident of 1994 in Hong Kong","1994",regex=False)
    .str.replace("Circa", "", regex=False)
    .str.replace("summer 1960", "1960", regex=False)
    .str.replace("Between May & Nov-1993", "1993", regex=False)
)
shark_attack_df = shark_attack_df[
    ~shark_attack_df["Date"].isin([
        ' 19-Jul-2004 to have happened  "on the weekend"',
        'Jul-1985 or mid Jul-1986',
        '1960-1961'
    ])
]

In [149]:
invalid_dates = shark_attack_df.loc[
    shark_attack_df["Date_parsed"].isna(),
    "Date"
]

invalid_dates.unique()

array(['24th May', '17th May', '16th May', '14th April', '3rd April',
       '26th March', '25th March', '18th March', '14th March',
       '10th March', '5th March', '22nd February', '6th February',
       '29th January', '24th January', '20th January', '19th January',
       '18th January', '13th January', '10th January', '8th January',
       '5th January', '3rd January ', '25th December', '21st December',
       '12th December', '9th December', '27th November ', '10th November',
       '9th November', '5th November', '4th November', '14th October',
       '11th October', '7th October', '29th September', '27th September',
       '6th September', '1st September', '30th August', '18th August',
       '17th August', '16th August', '7th August', '1st August',
       '28th July', '25th July', '22nd July', '20th July', '19th July',
       '18th July', '15th July', '6th July ', '6th July', '4th July',
       '29th June', '25th June', '22nd June', '17th June', '31st May',
       '02 Nov-202

In [150]:
shark_attack_df.sample(20)

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed
172,02 Oct-2023,2023,Unprovoked,USA,Surfing,F,Minor injury. Shark bit surfboard,N,10' to 12' Galapagos shark,2023-10-02
2126,30-Jun-2007,2007,Unprovoked,USA,Swimming,F,Hand bitten,N,NaN,2007-06-30
4461,05-Mar-1964,1964,Unprovoked,USA,Skin diving,M,"Minor injury, abdomen abraded when he collided...",N,1.7 m [5.5'] shark,1964-03-05
1027,02-Jun-2016,2016,Unprovoked,AUSTRALIA,Spearfishing,M,"No injury, but sharks repeatedly hit their fin...",NaN,Bronze whaler sharks x 3,2016-06-02
2405,06-Aug-2004,2004,Unprovoked,USA,Swimming,M,"Back, buttocks, left hand & left side of face ...",N,1.2 m [4'] shark,2004-08-06
3789,15-Jun-1981,1981,Watercraft,ENGLAND,Fishing,M,Minor injuries from shark that leapt aboard th...,N,"13', 400-lb thresher shark",NaT
1646,11-Aug--2011,2011,Unprovoked,USA,Swimming,M,Lower right leg bitten,N,NaN,NaT
4256,04-Feb-1968,1968,Unprovoked,AUSTRALIA,Spearfishing,M,"No injury, speargun bitten",N,"Bronze whaler shark, 3 m [10']",1968-02-04
2239,21-May-2006,2006,Unprovoked,BRAZIL,Surfing,M,"Injuries to left thigh, calf & foot",N,NaN,2006-05-21
4743,21-Aug-1960,1960,Unprovoked,USA,Standing in knee-deep water,M,"Lower right leg bitten, surgically amputated 1...",N,NaN,1960-08-21


In [151]:
#Clean 3 - Normalize characters to establish same format date
shark_attack_df["Date"] = (
    shark_attack_df["Date"]
    .astype(str)
    .str.strip()
    .str.replace("`", "", regex=False)
    .str.replace(".a", "", regex=False)
    .str.replace(".b", "", regex=False)
    .str.replace("--", "-", regex=False)
    .str.replace(".", "-", regex=False)
)
shark_attack_df.head()

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed
0,24th May,2026,Unprovoked,Australia,Spearfishing,M,Bite wounds to the head,Y,Undetermined Bull shark most likely,NaT
1,17th May,2026,Questionable,USA,Surfing,M,Bite wound to the hand,N,Blue fish bite most probable,NaT
2,16th May,2026,Unprovoked,Australia,Skindiving,M,Severe injuries both legs,Y,Great White Shark,NaT
3,14th April,2026,UNprovoked,Maldives,Swimming,M,Leg strpped off flesh later amputated in hospital,N,Unknown,NaT
4,3rd April,2026,Unprovoked,Australia,Surfing,M,Bite wound to R ankle,N,Bronze Whaler,NaT


In [152]:
dates_with_year = shark_attack_df["Date"][
    shark_attack_df["Date"].str.contains(r"\d{4}", na=False)
]

In [153]:
dates_with_year = (
    dates_with_year
    .str.strip()
    .str.replace("`", "", regex=False)
    .str.replace(".a", "", regex=False)
    .str.replace(".b", "", regex=False)
    .str.replace("--", "-", regex=False)
)

In [154]:
dates_with_year = (
    dates_with_year
    .str.replace(r"^(\d{2})\s+([A-Za-z]{3})-(\d{4})$", r"\1-\2-\3", regex=True)
    .str.replace(r"^(\d{2})\s+([A-Za-z]+)\s+(\d{4})$", r"\1-\2-\3", regex=True)
    .str.replace(r"^([A-Za-z]{3})-(\d{4})$", r"01-\1-\2", regex=True)
    .str.replace(r"^(\d{4})$", r"01-Jan-\1", regex=True)
)

In [155]:
parsed = pd.to_datetime(
    dates_with_year,
    errors="coerce"
)

dates_with_year[parsed.isna()].unique()

<ArrowStringArray>
[     '09 Sep- 2023',      '14-June 2023',       '08-Jun 2023',
       '28-May 2023',       '24-May 2023',       '21-May 2023',
      '19 -May 2023',      '09-Jul-2018-',          'May 2018',
        '2017-06-05',       '00-Feb-2017',      '3rd-Oct-2016',
          'May 2016',        '20-May2015',        '13-May2014',
        '29-Nov2013',     'December 2012',      '07-July-2012',
       '12-Jan 2011',       '190Feb-2010',      '31-July-2009',
        '2008-01-30',     'November 2011',             '2007-',
      '09-Jul-2006-',         'July 2006',      '24-Nov-2005-',
        '02-Ap-2001',      '13 -Nov-1999',          'May 1993',
       '04-Feb 1993',          'May 1992',         'July 1991',
      '03- Dec-1989',         'June 1983',       '24-May 1983',
          'May 1982',           'of 1981',        '12-30-1980',
             '1980s',          'May 1979',       '20-May 1975',
             '1970s', '13 or 30-May-1967',         'Sep- 1966',
           'of 1996',

In [156]:
dates_with_year = (
    dates_with_year

    # Espacios extra
    .str.strip()

    # Corregir separadores
    .str.replace("--", "-", regex=False)
    .str.replace(r"\s+", " ", regex=True)

    # Quitar guiones finales
    .str.replace(r"-$", "", regex=True)

    # Corregir meses abreviados incorrectos
    .str.replace("Ap-", "Apr-", regex=False)

    # Casos específicos
    .str.replace("190Feb-2010", "19-Feb-2010", regex=False)
    .str.replace("01-Jun-1018", "01-Jun-2018", regex=False)
    .str.replace("00-Feb-2017", "01-Feb-2017", regex=False)

    # Unir cuando falta espacio o guión
    .str.replace(r"(\d{2})-([A-Za-z]+)(\d{4})", r"\1-\2-\3", regex=True)

    # Convertir "May 2018" → "01-May-2018"
    .str.replace(r"^([A-Za-z]+)\s+(\d{4})$", r"01-\1-\2", regex=True)

    # Convertir "24-May 1983" → "24-May-1983"
    .str.replace(r"^(\d{1,2}-[A-Za-z]+)\s+(\d{4})$", r"\1-\2", regex=True)

    # Convertir "04-Feb 1993" → "04-Feb-1993"
    .str.replace(r"^(\d{1,2}-[A-Za-z]{3})\s+(\d{4})$", r"\1-\2", regex=True)

    # Convertir "09 Sep- 2023" → "09-Sep-2023"
    .str.replace(r"^(\d{1,2})\s+([A-Za-z]{3})-\s*(\d{4})$", r"\1-\2-\3", regex=True)

    # Convertir "14-June 2023" → "14-June-2023"
    .str.replace(r"^(\d{1,2})-([A-Za-z]+)\s+(\d{4})$", r"\1-\2-\3", regex=True)
)

In [157]:
parsed = pd.to_datetime(
    dates_with_year,
    errors="coerce"
)

dates_with_year[parsed.isna()].unique()

<ArrowStringArray>
[     '14-June-2023',      '19 -May 2023',        '2017-06-05',
      '3rd-Oct-2016',  '01-December-2012',      '07-July-2012',
      '31-July-2009',        '2008-01-30',  '01-November-2011',
              '2007',      '01-July-2006',      '13 -Nov-1999',
      '01-July-1991',      '03- Dec-1989',      '01-June-1983',
        '01-of-1981',        '12-30-1980',             '1980s',
             '1970s', '13 or 30-May-1967',         'Sep- 1966',
        '01-of-1996',      'May-Jun-1965',      'Jan-Jun-1962',
             '1960s']
Length: 25, dtype: str

In [158]:
#direct replacement (If not day or mont replace wit 01 Jan
replace_dict = {
    "14-June-2023": "14-Jun-2023",
    "19 -May 2023": "19-May-2023",
    "2017-06-05": "05-Jun-2017",
    "3rd-Oct-2016": "03-Oct-2016",
    "01-December-2012": "01-Dec-2012",
    "07-July-2012": "07-Jul-2012",
    "31-July-2009": "31-Jul-2009",
    "2008-01-30": "30-Jan-2008",
    "01-November-2011": "01-Nov-2011",
    "2007": "01-Jan-2007",
    "01-July-2006": "01-Jul-2006",
    "13 -Nov-1999": "13-Nov-1999",
    "01-July-1991": "01-Jul-1991",
    "03- Dec-1989": "03-Dec-1989",
    "01-June-1983": "01-Jun-1983",
    "01-of-1981": "01-Jan-1981",
    "12-30-1980": "30-Dec-1980",
    "1980s": "01-Jan-1980",
    "1970s": "01-Jan-1970",
    "13 or 30-May-1967": "21-May-1967",
    "Sep- 1966": "01-Sep-1966",
    "01-of-1996": "01-Jan-1996",
    "May-Jun-1965": "15-May-1965",
    "Jan-Jun-1962": "01-Jan-1962",
    "1960s": "01-Jan-1960"
}

dates_with_year = dates_with_year.replace(replace_dict)
dates_with_year.info()

<class 'pandas.Series'>
Index: 4640 entries, 137 to 4804
Series name: Date
Non-Null Count  Dtype
--------------  -----
4640 non-null   str  
dtypes: str(1)
memory usage: 122.3 KB


In [159]:
dates_with_year_2 = dates_with_year.copy()
pd.to_datetime(dates_with_year, format="%d-%b-%Y", errors="coerce").isna().sum()

np.int64(0)

In [160]:
#4 Work dates without the years
shark_attack_df.loc[dates_with_year.index, "Date"] = dates_with_year
shark_attack_df.head()

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed
0,24th May,2026,Unprovoked,Australia,Spearfishing,M,Bite wounds to the head,Y,Undetermined Bull shark most likely,NaT
1,17th May,2026,Questionable,USA,Surfing,M,Bite wound to the hand,N,Blue fish bite most probable,NaT
2,16th May,2026,Unprovoked,Australia,Skindiving,M,Severe injuries both legs,Y,Great White Shark,NaT
3,14th April,2026,UNprovoked,Maldives,Swimming,M,Leg strpped off flesh later amputated in hospital,N,Unknown,NaT
4,3rd April,2026,Unprovoked,Australia,Surfing,M,Bite wound to R ankle,N,Bronze Whaler,NaT


In [161]:
shark_attack_df.loc[
    ~shark_attack_df["Date"].astype(str).str.contains(r"\d{4}", na=False),
    "Date"
].unique()

<ArrowStringArray>
[       '24th May',        '17th May',        '16th May',      '14th April',
       '3rd April',      '26th March',      '25th March', '22nd-23rd March',
      '18th March',      '14th March',      '10th March',       '5th March',
   '22nd February',    '6th February',    '29th January',    '24th January',
    '20th January',    '19th January',    '18th January',    '13th January',
    '10th January',     '8th January',     '5th January',     '3rd January',
   '25th December',   '21st December',   '12th December',    '9th December',
   '27th November',   '10th November',    '9th November',    '5th November',
    '4th November',    '14th October',    '11th October',     '7th October',
  '29th September',  '27th September',   '6th September',   '1st September',
     '30th August',     '18th August',     '17th August',     '16th August',
      '7th August',      '1st August',       '28th July',       '25th July',
       '22nd July',       '20th July',       '19th July',

In [162]:
#5 Month adjustment
month_map = {
    "January": "Jan",
    "February": "Feb",
    "March": "Mar",
    "April": "Apr",
    "May": "May",
    "June": "Jun",
    "July": "Jul",
    "August": "Aug",
    "September": "Sep",
    "October": "Oct",
    "November": "Nov",
    "December": "Dec"
}

# Filas sin año en Date
mask = ~shark_attack_df["Date"].astype(str).str.contains(r"\d{4}", na=False)

# Eliminar st, nd, rd, th
shark_attack_df.loc[mask, "Date"] = (
    shark_attack_df.loc[mask, "Date"]
    .str.replace(r"(\d+)(st|nd|rd|th)", r"\1", regex=True)
)

# Extraer día y mes
temp = shark_attack_df.loc[mask, "Date"].str.extract(
    r"(\d+)\s+([A-Za-z]+)"
)

# Convertir a formato DD-MMM
shark_attack_df.loc[mask, "Date"] = (
    temp[0].str.zfill(2)
    + "-"
    + temp[1].map(month_map)
)

In [163]:
shark_attack_df.loc[mask, "Date"] = (
    shark_attack_df.loc[mask, "Date"]
    + "-"
    + shark_attack_df.loc[mask, "Year"].astype(int).astype(str)
)

In [164]:
shark_attack_df.loc[mask, "Date"] = (
    shark_attack_df.loc[mask, "Date"]
    + "-"
    + shark_attack_df.loc[mask, "Year"].astype(int).astype(str)
)
shark_attack_df.head()

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed
0,24-May-2026-2026,2026,Unprovoked,Australia,Spearfishing,M,Bite wounds to the head,Y,Undetermined Bull shark most likely,NaT
1,17-May-2026-2026,2026,Questionable,USA,Surfing,M,Bite wound to the hand,N,Blue fish bite most probable,NaT
2,16-May-2026-2026,2026,Unprovoked,Australia,Skindiving,M,Severe injuries both legs,Y,Great White Shark,NaT
3,14-Apr-2026-2026,2026,UNprovoked,Maldives,Swimming,M,Leg strpped off flesh later amputated in hospital,N,Unknown,NaT
4,03-Apr-2026-2026,2026,Unprovoked,Australia,Surfing,M,Bite wound to R ankle,N,Bronze Whaler,NaT


In [165]:
#format stablished for the changes rows and NAT Caleaning
shark_attack_df["Date"] = pd.to_datetime(
    shark_attack_df["Date"],
    format="%d-%b-%Y",
    errors="coerce"
)

In [166]:
shark_attack_df["Date"].isna().sum()

np.int64(161)

In [167]:
shark_attack_df.loc[
    ~shark_attack_df["Date"].astype(str).str.contains(r"\d{4}", na=False),
    "Date"
].unique()

<DatetimeArray>
['NaT']
Length: 1, dtype: datetime64[us]

In [168]:
mask = shark_attack_df["Date"].isna()

shark_attack_df.loc[mask, "Date"] = pd.to_datetime(
    "01-Jan-" +
    shark_attack_df.loc[mask, "Year"].astype(int).astype(str),
    format="%d-%b-%Y"
)

In [169]:
shark_attack_df["Date"].isna().sum()

np.int64(0)

In [170]:
shark_attack_df["Date"].dtype

dtype('<M8[us]')

In [171]:
shark_attack_df["Year"] = pd.to_numeric(
    shark_attack_df["Year"],
    errors="coerce"
)

In [172]:
shark_attack_df["Date_Year"] = (
    shark_attack_df["Date"].dt.year
)

In [173]:
(
    shark_attack_df["Date"].dt.year
    !=
    shark_attack_df["Year"]
).sum()

np.int64(16)

In [174]:
#7 Drop colums with diferent number of age form date and year columns

shark_attack_df = shark_attack_df[
    shark_attack_df["Date"].dt.year == shark_attack_df["Year"]
]
(
    shark_attack_df["Date"].dt.year
    !=
    shark_attack_df["Year"]
).sum()

np.int64(0)

In [175]:
shark_attack_df.info()

<class 'pandas.DataFrame'>
Index: 4785 entries, 0 to 4804
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         4785 non-null   datetime64[us]
 1   Year         4785 non-null   int64         
 2   Type         4771 non-null   str           
 3   Country      4773 non-null   str           
 4   Activity     4485 non-null   str           
 5   Sex          4439 non-null   str           
 6   Injury       4764 non-null   str           
 7   Fatal Y/N    4439 non-null   object        
 8   Species      3106 non-null   str           
 9   Date_parsed  4483 non-null   datetime64[ns]
 10  Date_Year    4785 non-null   int32         
dtypes: datetime64[ns](1), datetime64[us](1), int32(1), int64(1), object(1), str(6)
memory usage: 780.3+ KB


**COUNTRY COLUMN CLEANING**

In [176]:
shark_attack_df["Country"].isna().sum()

np.int64(12)

In [177]:
shark_attack_df[
    shark_attack_df["Country"].astype(str).str.strip() == ""
]

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed,Date_Year


In [178]:
shark_attack_df.loc[
    shark_attack_df["Country"].isna()
]

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed,Date_Year
814,2017-11-13,2017,Unprovoked,NaN,Surfing,M,Puncture wounds to feet,N,NaN,NaT,2017
1283,2014-08-01,2014,Invalid,NaN,Sea disaster,M,Shark involvement prior to death not confirmed,NaN,Shark involvement not confirmed,2014-08-01,2014
3719,1983-01-01,1983,Unprovoked,NaN,Swimming,M,Left leg bitten,N,NaN,NaT,1983
4149,1970-11-01,1970,Unprovoked,NaN,NaN,M,Extensive injuries,N,NaN,1970-11-01,1970
4158,1970-08-02,1970,Invalid,NaN,Sea Disaster Sinking of ferryboat Christina,NaN,"Sharks scavenged on bodies, but no record of t...",NaN,Shark involvement prior to death was not confi...,1970-08-02,1970
4159,1970-07-05,1970,Unprovoked,NaN,NaN,M,Finger or toe severed,N,Mako shark,1970-07-05,1970
4166,1970-04-01,1970,Provoked,NaN,Freediving,M,Arm abraded & lacerated. Recorded as PROVOKED ...,N,Wobbegong shark,1970-04-01,1970
4170,1970-02-05,1970,Unprovoked,NaN,Wading,F,Lacerations to lower leg,N,Carpet shark,1970-02-05,1970
4196,1969-08-01,1969,Unprovoked,NaN,NaN,M,Am lacerated,N,NaN,1969-08-01,1969
4376,1965-10-21,1965,Unprovoked,NaN,The boat Caribou II sank,M,Survived,N,NaN,1965-10-21,1965


In [179]:
#NaN Drop - Not relevant data

shark_attack_df = shark_attack_df.dropna(subset=["Country"])
shark_attack_df["Country"].isna().sum()

np.int64(0)

In [180]:
#Lower case and look for the ones that are not countries
shark_attack_df["Country"] = (
    shark_attack_df["Country"]
    .str.lower()
    .str.strip()
)

In [181]:
sorted(shark_attack_df["Country"].unique())

['admiralty islands',
 'angola',
 'antigua',
 'argentina',
 'aruba',
 'atlantic ocean',
 'australia',
 'azores',
 'bahamas',
 'bangladesh',
 'belize',
 'bermuda',
 'brazil',
 'british isles',
 'british overseas territory',
 'british virgin islands',
 'british west indies',
 'canada',
 'canary islands',
 'cape verde',
 'caribbean sea',
 'cayman islands',
 'chile',
 'china',
 'colombia',
 'columbia',
 'comoros',
 'coral sea',
 'costa rica',
 'croatia',
 'cuba',
 'diego garcia',
 'dominican republic',
 'ecuador',
 'egypt',
 'egypt / israel',
 'el salvador',
 'england',
 'federated states of micronesia',
 'fiji',
 'france',
 'french polynesia',
 'grand cayman',
 'greece',
 'grenada',
 'guam',
 'gulf of aden',
 'hawaii',
 'honduras',
 'hong kong',
 'india',
 'indonesia',
 'iran',
 'iraq',
 'ireland',
 'israel',
 'italy',
 'jamaica',
 'japan',
 'johnston island',
 'jordan',
 'kenya',
 'kiribati',
 'liberia',
 'libya',
 'madagascar',
 'malaysia',
 'maldive islands',
 'maldives',
 'malta',
 'm

In [182]:
countries = [['angola', 'antigua and barbuda', 'argentina', 'aruba', 'australia', 'bahamas', 'bangladesh', 'belize', 'bermuda', 'brazil', 'cabo verde', 'canada', 'cayman islands', 'chile', 'china', 'colombia', 'comoros', 'coral sea', 'costa rica', 'croatia', 'cuba', 'dominican republic', 'ecuador', 'egypt', 'el salvador', 'fiji', 'france', 'french polynesia', 'grand cayman', 'greece', 'grenada', 'guam', 'honduras', 'hong kong', 'india', 'indonesia', 'iran', 'iraq', 'ireland', 'israel', 'italy', 'jamaica', 'japan', 'johnston island', 'jordan', 'kenya', 'kiribati', 'liberia', 'libya', 'madagascar', 'malaysia', 'maldives', 'malta', 'marshall islands', 'mauritius', 'mexico', 'micronesia', 'montenegro', 'morocco', 'mozambique', 'namibia', 'new britain', 'new caledonia', 'new guinea', 'new zealand', 'nicaragua', 'nigeria', 'north pacific ocean', 'norway', 'palau', 'panama', 'papua new guinea', 'philippines', 'portugal', 'puerto rico', 'red sea', 'red sea / indian ocean', 'russian federation', 'saint kitts and nevis', 'saint martin', 'samoa', 'saudi arabia', 'senegal', 'seychelles', 'sierra leone', 'singapore', 'solomon islands', 'somalia', 'south africa', 'south korea', 'south pacific ocean', 'spain', 'sri lanka', 'st helena, british overseas territory', 'sudan', 'taiwan', 'tanzania', 'thailand', 'tonga', 'trinidad and tobago', 'tunisia', 'turkiye', 'turks and caicos islands', 'united arab emirates', 'united kingdom', 'united states', 'uruguay', 'vanuatu', 'venezuela', 'vietnam', 'yemen']]

In [183]:
invalid_countries = shark_attack_df[
    ~shark_attack_df["Country"].isin(countries)
]["Country"].unique()

print(invalid_countries)

<ArrowStringArray>
[                     'australia',                            'usa',
                       'maldives',                        'bahamas',
                  'new caledonia',                 'cayman islands',
                         'brazil',              'us virgin islands',
                     'mozambique',               'french polynesia',
 ...
            'north pacific ocean', 'federated states of micronesia',
             'mid atlantic ocean',              'admiralty islands',
            'british west indies',           'south atlantic ocean',
                   'persian gulf',         'red sea / indian ocean',
                      'north sea',                      'nicaragua']
Length: 150, dtype: str


In [184]:
len(invalid_countries)

150

In [185]:
print(countries)

[['angola', 'antigua and barbuda', 'argentina', 'aruba', 'australia', 'bahamas', 'bangladesh', 'belize', 'bermuda', 'brazil', 'cabo verde', 'canada', 'cayman islands', 'chile', 'china', 'colombia', 'comoros', 'coral sea', 'costa rica', 'croatia', 'cuba', 'dominican republic', 'ecuador', 'egypt', 'el salvador', 'fiji', 'france', 'french polynesia', 'grand cayman', 'greece', 'grenada', 'guam', 'honduras', 'hong kong', 'india', 'indonesia', 'iran', 'iraq', 'ireland', 'israel', 'italy', 'jamaica', 'japan', 'johnston island', 'jordan', 'kenya', 'kiribati', 'liberia', 'libya', 'madagascar', 'malaysia', 'maldives', 'malta', 'marshall islands', 'mauritius', 'mexico', 'micronesia', 'montenegro', 'morocco', 'mozambique', 'namibia', 'new britain', 'new caledonia', 'new guinea', 'new zealand', 'nicaragua', 'nigeria', 'north pacific ocean', 'norway', 'palau', 'panama', 'papua new guinea', 'philippines', 'portugal', 'puerto rico', 'red sea', 'red sea / indian ocean', 'russian federation', 'saint kit

In [186]:
#2 Replace manual for real countries in the country list
replace_dict = {
    #Countries
        "usa": "united states",
        "columbia": "colombia",
        "maldive islands": "maldives",
        "cape verde": "cabo verde",
        "south korea": "south korea",
        "vietnam": "vietnam",
        "russia": "russian federation",
        "tanzania": "tanzania",
        "venezuela": "venezuela",
        "iran": "iran",
        "turkey": "turkiye",
        "western samoa": "samoa",
        "trinidad": "trinidad and tobago",
        "tobago": "trinidad and tobago",
        "trinidad & tobago": "trinidad and tobago",
        "antigua": "antigua and barbuda",
        "hawaii": "united states",
        "england": "united kingdom",
        "scotland": "united kingdom",
        "british isles": "united kingdom",
        "canary islands": "spain",
        "azores": "portugal",
        "reunion": "france",
        "reunion island": "france",
        "okinawa": "japan",
        "us virgin islands": "united states",
        "british virgin islands": "united kingdom",
        "turks & caicos": "turks and caicos islands",
        "turks and caicos": "turks and caicos islands",
        "st martin": "saint martin",
        "st. martin": "saint martin",
        "st. maartin": "sint maarten",
        "nevis": "saint kitts and nevis",
        "st kitts / nevis": "saint kitts and nevis",
        "united arab emirates (uae)": "united arab emirates",
        "federated states of micronesia": "micronesia",
        "south china sea" : "china",
        "new guinea": "papua new guinea",
        "new britain" : "papua new guinea",
        "grenada" : "trinidad and tobago"

    }

shark_attack_df["Country"] = (
    shark_attack_df["Country"]
    .replace(replace_dict)
)

In [187]:
invalid_countries_2 = shark_attack_df[
    ~shark_attack_df["Country"].isin(countries)
]["Country"].unique()

print(invalid_countries_2)

<ArrowStringArray>
[             'australia',          'united states',               'maldives',
                'bahamas',          'new caledonia',         'cayman islands',
                 'brazil',             'mozambique',       'french polynesia',
                  'samoa',
 ...
                'red sea',    'north pacific ocean',     'mid atlantic ocean',
      'admiralty islands',    'british west indies',   'south atlantic ocean',
           'persian gulf', 'red sea / indian ocean',              'north sea',
              'nicaragua']
Length: 125, dtype: str


In [188]:
shark_attack_df[
    shark_attack_df["Country"].isin(invalid_countries_2)
]["Country"].value_counts()

Country
united states             2096
australia                  892
south africa               451
bahamas                    123
brazil                     114
                          ... 
british west indies          1
south atlantic ocean         1
red sea / indian ocean       1
north sea                    1
nicaragua                    1
Name: count, Length: 125, dtype: int64

In [189]:
#Not countries

not_country = ["atlantic ocean" ,
              "caribbean sea",
              "pacific ocean" ,
              "mid atlantic ocean" ,
              "persian gulf" ,
              "st helena",
              "british overseas territory" ,
              "admiralty islands",
              "british west indies",
              "south atlantic ocean",
              "red sea coral sea ",
              "northern arabian sea",
              "north atlantic ocean",
              "british overseas territory"
              "egypt / israel",
              "sint maarten",
              "gulf of aden",
              "palestinian territories",
              "diego garcia",
              "north sea",
              "south pacific ocean",
              "red sea / indian ocean",
              "red sea",
              "coral sea",
              "guam",
              "johnston island",
              "kiribati",
              "north pacific ocean",
              "british overseas territory"


]
shark_attack_df = shark_attack_df[
    ~shark_attack_df["Country"].isin(not_country)
]

In [190]:
shark_attack_df["Country"].value_counts()

Country
united states    2096
australia         892
south africa      451
bahamas           123
brazil            114
                 ... 
namibia             1
bangladesh          1
singapore           1
sudan               1
nicaragua           1
Name: count, Length: 101, dtype: int64

In [191]:
countries_df = sorted(
    shark_attack_df["Country"]
    .dropna()
    .unique()
    .tolist()
)

print(countries_df)

['angola', 'antigua and barbuda', 'argentina', 'aruba', 'australia', 'bahamas', 'bangladesh', 'belize', 'bermuda', 'brazil', 'cabo verde', 'canada', 'cayman islands', 'chile', 'china', 'colombia', 'comoros', 'costa rica', 'croatia', 'cuba', 'dominican republic', 'ecuador', 'egypt', 'egypt / israel', 'el salvador', 'fiji', 'france', 'french polynesia', 'grand cayman', 'greece', 'honduras', 'hong kong', 'india', 'indonesia', 'iran', 'iraq', 'ireland', 'israel', 'italy', 'jamaica', 'japan', 'jordan', 'kenya', 'liberia', 'libya', 'madagascar', 'malaysia', 'maldives', 'malta', 'marshall islands', 'mauritius', 'mexico', 'micronesia', 'montenegro', 'morocco', 'mozambique', 'namibia', 'new caledonia', 'new zealand', 'nicaragua', 'nigeria', 'norway', 'palau', 'panama', 'papua new guinea', 'philippines', 'portugal', 'puerto rico', 'russian federation', 'saint kitts and nevis', 'saint martin', 'samoa', 'saudi arabia', 'senegal', 'seychelles', 'sierra leone', 'singapore', 'solomon islands', 'somal

In [192]:
shark_attack_df.info()

<class 'pandas.DataFrame'>
Index: 4732 entries, 0 to 4804
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         4732 non-null   datetime64[us]
 1   Year         4732 non-null   int64         
 2   Type         4718 non-null   str           
 3   Country      4732 non-null   str           
 4   Activity     4439 non-null   str           
 5   Sex          4395 non-null   str           
 6   Injury       4711 non-null   str           
 7   Fatal Y/N    4393 non-null   object        
 8   Species      3078 non-null   str           
 9   Date_parsed  4437 non-null   datetime64[ns]
 10  Date_Year    4732 non-null   int32         
dtypes: datetime64[ns](1), datetime64[us](1), int32(1), int64(1), object(1), str(6)
memory usage: 790.4+ KB


**SEX COLUMN CLEANING**

In [193]:
# STATUS BEFORE THE CLEANING

shark_attack_df["Sex"].value_counts(dropna=False)

Sex
M      3702
F       681
NaN     337
M         5
?         2
F         2
 M        1
m         1
lli       1
Name: count, dtype: int64

In [194]:
# Remove extra spaces

shark_attack_df["Sex"] = shark_attack_df["Sex"].str.strip()

In [195]:
# Standardize uppercase / lowercase

shark_attack_df["Sex"] = shark_attack_df["Sex"].str.upper()

In [196]:
# Replace M * 2 with M

shark_attack_df["Sex"] = shark_attack_df["Sex"].replace({
    "M X 2": "M"
})

In [197]:
# Replace unclear values with Unknown

shark_attack_df.loc[
    (shark_attack_df["Sex"] != "M") &
    (shark_attack_df["Sex"] != "F"),
    "Sex"
] = "Unknown"

In [198]:
# STATUS AFTER THE CLEANING

shark_attack_df["Sex"].value_counts(dropna=False)

Sex
M          3709
F           683
Unknown     340
Name: count, dtype: int64

**TYPE COLUMN CLEANING**

In [199]:
# STATUS BEFORE CLEANING

shark_attack_df["Type"].value_counts(dropna=False)

Type
Unprovoked             3632
Provoked                425
Invalid                 337
Watercraft              216
Sea Disaster             73
Questionable             27
NaN                      14
 Provoked                 2
UNprovoked                1
unprovoked                1
?                         1
Unconfirmed               1
Unverified                1
Under investigation       1
Name: count, dtype: int64

In [200]:
# Remove extra spaces

shark_attack_df["Type"] = shark_attack_df["Type"].str.strip()

In [201]:
# Standardize capitalization

shark_attack_df["Type"] = shark_attack_df["Type"].replace({
    "unprovoked": "Unprovoked",
    "UNprovoked": "Unprovoked",
    "Provoked ": "Provoked"
})

In [202]:
# Handling missing values

shark_attack_df["Type"] = shark_attack_df["Type"].replace({
    "?": "Unknown",
    "Unconfirmed": "Unknown",
    "Unverified": "Unknown",
    "Under investigation": "Unknown"
})

shark_attack_df["Type"] = shark_attack_df["Type"].fillna("Unknown")

In [203]:
# Combine "boat" with "watercraft"

shark_attack_df["Type"] = shark_attack_df["Type"].replace({
    "Boat": "Watercraft"
})

In [204]:
shark_attack_df["Type"].value_counts(dropna=False)

Type
Unprovoked      3634
Provoked         427
Invalid          337
Watercraft       216
Sea Disaster      73
Questionable      27
Unknown           18
Name: count, dtype: int64

**ACTIVITY COLUMN CLEANING**

In [205]:
# STATUS BEFORE THE CLEANING

shark_attack_df["Activity"].value_counts(dropna=False).head(30)

Activity
Surfing            1135
Swimming            638
Spearfishing        338
NaN                 293
Fishing             278
Wading              148
Snorkeling          133
Diving               99
Standing             82
Scuba diving         78
Body boarding        62
Boogie boarding      42
Kayaking             40
Body surfing         33
Free diving          23
Windsurfing          20
Boogie Boarding      18
Swimming             17
Scuba Diving         13
Bathing              13
Playing              13
Walking              13
Surf skiing          12
Treading water       12
Surf-skiing          12
Shark fishing        12
Paddle boarding      11
Surf fishing         11
Fishing              11
Freediving           10
Name: count, dtype: int64

In [206]:
# Unique activities

shark_attack_df["Activity"].nunique()

863

In [207]:
# Remove extra spaces

shark_attack_df["Activity"] = (
    shark_attack_df["Activity"]
    .str.strip()
)

In [208]:
# Convert to lowercase

shark_attack_df["Activity"] = (
    shark_attack_df["Activity"]
    .str.lower()
)

In [209]:
# Handling the missing values

shark_attack_df["Activity"].isna().sum()

np.int64(293)

In [210]:
# Handling the missing values

shark_attack_df["Activity"] = (
    shark_attack_df["Activity"]
    .fillna("unknown")
)

In [211]:
# STATUS AFTER CLEANING

shark_attack_df["Activity"].value_counts().head(30)

Activity
surfing            1140
swimming            655
spearfishing        340
unknown             296
fishing             289
wading              148
snorkeling          134
diving              101
scuba diving        100
standing             83
body boarding        69
boogie boarding      60
kayaking             43
body surfing         38
free diving          25
windsurfing          20
surf skiing          18
kayak fishing        15
playing              14
bathing              13
walking              13
shark fishing        13
paddle boarding      12
kite surfing         12
treading water       12
floating             12
surf-skiing          12
surf fishing         11
sea disaster         11
freediving           10
Name: count, dtype: int64

**CREATING AN ACTIVITY GROUP COLUMN FOR FURTHER CLEANING OF THE ACTIVITY**

In [212]:
def activity_group(activity):

    if activity == "unknown":
        return "Unknown"

    elif "surf" in activity:
        return "Surfing"

    elif "body boarding" in activity:
        return "Surfing"

    elif "boogie boarding" in activity:
        return "Surfing"

    elif "swim" in activity:
        return "Swimming"

    elif "bathing" in activity:
        return "Swimming"

    elif "treading water" in activity:
        return "Swimming"

    elif "fish" in activity:
        return "Fishing"

    elif "dive" in activity:
        return "Diving"

    elif "snork" in activity:
        return "Diving"

    elif "kayak" in activity:
        return "Boating/Paddling"

    elif "canoe" in activity:
        return "Boating/Paddling"

    elif "rowing" in activity:
        return "Boating/Paddling"

    elif "fell overboard" in activity:
        return "Boating/Paddling"

    elif "wading" in activity:
        return "Shore Activities"

    elif "standing" in activity:
        return "Shore Activities"

    elif "walking" in activity:
        return "Shore Activities"

    else:
        return "Other"

In [213]:
shark_attack_df["Activity_Group"] = (
    shark_attack_df["Activity"]
    .apply(activity_group)
)

In [214]:
shark_attack_df["Activity_Group"].value_counts()

Activity_Group
Surfing             1522
Fishing              886
Other                779
Swimming             774
Unknown              296
Shore Activities     264
Diving               141
Boating/Paddling      70
Name: count, dtype: int64

In [215]:
#Checking activity information in the "Injury" to decrease number of Unknown data points

unknown_activity_df = shark_attack_df[
    shark_attack_df["Activity"] == "unknown"
]

unknown_activity_df.shape

(296, 12)

In [216]:
unknown_activity_df["Injury"].sample(30)

1977                                       Minor injuries
2830                                    Right calf bitten
3150                        Puncture wounds on right foot
4015                                             Survived
4135                            FATAL, body not recovered
2328    A hoax - No shark was involved and Wells' "dau...
4305                                           Leg bitten
3331                      Survived. questionable incident
1490                                    Laceration to toe
4136                            FATAL, body not recovered
1616         Lacerations to right wrist and middle finger
4104                                      Left leg bitten
3087                                           No details
3666    Human remains recovered, evidence of scavengin...
1386        Arm bitten by captive shark PROVOKED INCIDENT
291                                        minor injuries
1640                               Abrasions to left hand
4581          

In [217]:
#Checking activity information in the "Injury" column using regex

pattern = r"\b(?:surf|surfing|body surfing|windsurfing)\b"

unknown_activity_df["Injury"].str.contains(
    pattern,
    case=False,
    regex=True,
    na=False
).sum()

np.int64(0)

In [218]:
pattern = r"\b(?:swim|swimming|bathing|treading water)\b"

unknown_activity_df["Injury"].str.contains(
    pattern,
    case=False,
    regex=True,
    na=False
).sum()

np.int64(1)

In [219]:
pattern = r"\b(?:fish|fishing|spearfishing)\b"

unknown_activity_df["Injury"].str.contains(
    pattern,
    case=False,
    regex=True,
    na=False
).sum()

np.int64(0)

In [220]:
pattern = r"\b(?:surf|surfing|body surfing|windsurfing)\b"

unknown_activity_df[
    unknown_activity_df["Injury"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )
][["Injury"]]

,Injury


In [221]:
unknown_activity_df[
    ["Activity", "Injury"]
].sample(50)

,Activity,Injury
4418,unknown,FATAL. Said to have been killed by sorcerers t...
486,unknown,Minor lacerations to right foot when he steppe...
1418,unknown,Probable drowning with post-mortem bites
2387,unknown,No injury
322,unknown,Cut to knee
1386,unknown,Arm bitten by captive shark PROVOKED INCIDENT
528,unknown,NaN
3857,unknown,FATAL
3127,unknown,Hand bitten (minor injury)
4144,unknown,Survived


In [222]:
# We checked whether the Injury column could help recover missing Activity values.
# Only a very small number (3) of unknown activities had useful activity clues.
# Because the impact is not material, we keep these records as "Unknown".

**FATAL Y/N COLUMN CLEANING**

In [223]:
#Cleaning the Fatal Y/N column
#Checking the unique values on the column
shark_attack_df["Fatal Y/N"].unique()

array(['Y', 'N', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, ' N'],
      dtype=object)

In [224]:
#Standardizing font size
shark_attack_df["Fatal Y/N"] =  shark_attack_df["Fatal Y/N"].str.lower()

In [225]:
#The value "n" is duplicated because there are spaces. Let's remove it.
shark_attack_df["Fatal Y/N"] = shark_attack_df["Fatal Y/N"].str.strip()
shark_attack_df["Fatal Y/N"].unique()

array(['y', 'n', 'f', 'm', nan, 'nq', 'unknown'], dtype=object)

In [226]:
#investigating the number of cases where the values ​​are not "y" and "n"
shark_attack_df.loc[shark_attack_df["Fatal Y/N"] == "f"]

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed,Date_Year,Activity_Group
122,2024-01-01,2024,Unprovoked,united states,surfing,M,Not specific shark bite injuries to the body,f,Not stated,2024-06-23,2024,Surfing
245,2023-02-19,2023,Unprovoked,new caledonia,swimming,M,"FATAL, Multiple injuries",f,Bull shark,2023-02-19,2023,Swimming
573,2019-12-12,2019,Questionable,france,kayaking,M,Partial remains recovered 12/26/2019 from shark,f,3.4 m tiger shark,2019-12-12,2019,Boating/Paddling
579,2019-11-23,2019,Questionable,australia,spearfishing,M,Believed to have drowned. Partial remains wash...,f,NaN,2019-11-23,2019,Fishing
955,2016-11-17,2016,Unprovoked,mozambique,body recovery,M,Body washed up on beach with signs of shark at...,f,Unknown,2016-11-17,2016,Other


In [227]:
#As indicated on the incident reporting website, all individuals survived unless noted otherwise.
#Since we don't have the information in the "Fatal Y/N" column, we'll check the information in the "Injury" column.

In [228]:
#Analized manually, info of fatal attack in column "injury" of the value "f". By the info in Injury it means fatal.
shark_attack_df["Fatal Y/N"] = shark_attack_df["Fatal Y/N"].replace({
    "f": "n"
})

In [229]:
shark_attack_df.columns

Index(['Date', 'Year', 'Type', 'Country', 'Activity', 'Sex', 'Injury',
       'Fatal Y/N', 'Species ', 'Date_parsed', 'Date_Year', 'Activity_Group'],
      dtype='str')

In [230]:
shark_attack_df["Fatal Y/N"].unique()

array(['y', 'n', 'm', nan, 'nq', 'unknown'], dtype=object)

In [231]:
shark_attack_df["Fatal Y/N"].loc[shark_attack_df["Fatal Y/N"] == "f"]

Series([], Name: Fatal Y/N, dtype: object)

In [232]:
shark_attack_df.loc[shark_attack_df["Fatal Y/N"] == "m"]

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed,Date_Year,Activity_Group
179,2023-09-09,2023,Unprovoked,philippines,fishing,M,Laceration to left thigh,m,NaN,NaT,2023,Fishing
329,2022-04-01,2022,Unknown,south africa,unknown,M,Possible drowing and scavenging,m,NaN,2022-04-01,2022,Unknown
1044,2016-04-18,2016,Provoked,french polynesia,spearfishing,M,Laceration to knee by speared shark PROVOKED I...,m,"Grey reef shark, 2 m",2016-04-18,2016,Fishing


In [233]:
#Analized manually, info of fatal attack in column "injury" of the value "m". By the info in Injury it does not means fatal by shark.
shark_attack_df["Fatal Y/N"] = shark_attack_df["Fatal Y/N"].replace({
    "m": "n"
})

In [234]:
shark_attack_df["Fatal Y/N"].unique()

array(['y', 'n', nan, 'nq', 'unknown'], dtype=object)

In [235]:
#Continuing to explore the values
shark_attack_df.loc[shark_attack_df["Fatal Y/N"] == "nq"]

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed,Date_Year,Activity_Group
406,2021-06-28,2021,Unprovoked,united states,surfing,F,Lacerations to thigh and calf,nq,NaN,2021-06-28,2021,Surfing


In [ ]:
#Analized manually, info of fatal attack in column "injury" of the value "nq". By the info in Injury it means not fatal.
shark_attack_df["Fatal Y/N"] = shark_attack_df["Fatal Y/N"].replace({
    "nq": "n"
})

In [237]:
shark_attack_df["Fatal Y/N"].unique()

array(['y', 'n', nan, 'unknown'], dtype=object)

In [238]:
shark_attack_df[shark_attack_df["Fatal Y/N"].isna()]

,Date,Year,Type,Country,Activity,Sex,Injury,Fatal Y/N,Species,Date_parsed,Date_Year,Activity_Group
183,2023-08-25,2023,Unprovoked,australia,surfing,M,Severe injuries to lower limbs,NaN,"White shark, 3.8-4.2m",2023-08-25,2023,Surfing
186,2023-08-21,2023,Questionable,bahamas,unknown,M,Body found with shark bites. Possible drowning...,NaN,NaN,2023-08-21,2023,Unknown
211,2023-06-07,2023,Unprovoked,bahamas,scuba diving,F,Calf severely bitten,NaN,Caribbean rreef shark,2023-06-07,2023,Other
243,2023-03-02,2023,Unprovoked,seychelles,snorkeling,M,Left foot bitten,NaN,Lemon shark,2023-03-02,2023,Diving
247,2023-02-18,2023,Questionable,argentina,unknown,M,Death by misadventure,NaN,NaN,2023-02-18,2023,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...
4744,1960-08-14,1960,Invalid,mozambique,boat with 46 people on board capsized,Unknown,1 survivor,NaN,Shark involvement not confirmed,1960-08-14,1960,Other
4757,1960-06-07,1960,Invalid,united states,testing classified underwater electronic gear ...,M,"Legs & arms bitten, coroner unable to determin...",NaN,Shark involvement prior to death was not confi...,1960-06-07,1960,Other
4769,1960-04-14,1960,Invalid,bermuda,floating on a raft,M,"No injury, 5 sharks bumped raft",NaN,Questionable incident,1960-04-14,1960,Other
4782,1960-02-27,1960,Invalid,united states,spearfishing,M,Left forearm lacerated,NaN,"According to Benjamin, the injury was inflicte...",1960-02-27,1960,Fishing


In [239]:
#Checking fatality information in the "Injury" column using regex
pattern = r"\b(fatal\w*|lethal\w*|deadly\w*|mortal\w*|terminal\w*)\b"

shark_attack_df["Contains_Fatal"] = shark_attack_df["Injury"].str.contains(pattern, case=False, regex=True)

shark_attack_df["Contains_Fatal"]

/tmp/ipykernel_10728/474157297.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  shark_attack_df["Contains_Fatal"] = shark_attack_df["Injury"].str.contains(pattern, case=False, regex=True)


0       False
1       False
2       False
3       False
4       False
        ...  
4799    False
4801     True
4802    False
4803     True
4804    False
Name: Contains_Fatal, Length: 4732, dtype: bool

In [240]:
shark_attack_df["Fatal Y/N"] = shark_attack_df["Fatal Y/N"].fillna(
    shark_attack_df["Contains_Fatal"].apply(lambda x: "y" if x else "n")
)

In [241]:
shark_attack_df["Fatal Y/N"].unique()

array(['y', 'n', 'unknown'], dtype=object)

In [242]:
shark_attack_df.loc[shark_attack_df["Fatal Y/N"].isin(["unknown"]), "Fatal Y/N"] = \
    shark_attack_df["Contains_Fatal"].apply(lambda x: "y" if x else "n")

In [243]:
shark_attack_df["Fatal Y/N"].unique()

array(['y', 'n'], dtype=object)

In [244]:
shark_attack_df["Fatal Y/N"].value_counts()

Fatal Y/N
n    4124
y     608
Name: count, dtype: int64

**SPECIES COLUMN CLEANING**

In [245]:
#The column title has a space at the end. Let's correct it.
shark_attack_df.rename(columns={"Species " : "specie"}, inplace=True)

In [246]:
shark_attack_df.columns

Index(['Date', 'Year', 'Type', 'Country', 'Activity', 'Sex', 'Injury',
       'Fatal Y/N', 'specie', 'Date_parsed', 'Date_Year', 'Activity_Group',
       'Contains_Fatal'],
      dtype='str')

In [247]:
shark_attack_df["specie"].unique()

<ArrowStringArray>
[                                        'Undetermined Bull shark most likely',
                                                'Blue fish bite most probable',
                                                           'Great White Shark',
                                                                     'Unknown',
                                                               'Bronze Whaler',
                                             'Great White Shark 3.5m (11.5ft)',
                                                    'Unknown shark 5ft (1.5m)',
                                                 'Great White Shark 10ft (3m)',
                                              'Tiger or bull shark implicated',
                                             'Juvenile Tiger shark 7ft (2.1m)',
 ...
                                              'Bronze whaler shark,4 m [13'] ',
 'According to Benjamin, the injury was inflicted by a barracuda, not a shark',
                

In [248]:
#Creating a function to search for patterns that include "shark" and extracting it
def extract_shark_name(text):
    # Convert the input to string, handle NaN as empty string
    text = str(text) if not pd.isna(text) else ""

    # Define regex pattern to match shark names
    match = re.search(r'(\b\w*\s*\w*shark\b)', text, re.I)
    return match.group(0) if match else np.nan

In [249]:
shark_attack_df["shark_name"] = shark_attack_df["specie"].apply(extract_shark_name)

In [250]:
shark_attack_df["shark_name"] = shark_attack_df["shark_name"].str.lower()

In [251]:
shark_attack_df["shark_name"].unique()

<ArrowStringArray>
[        'bull shark',                  nan,        'white shark',
      'unknown shark',        'tiger shark',         'gill shark',
           '5m shark',         'reef shark',           '3m shark',
        'nurse shark',           '2m shark',        'lemon shark',
        'shall shark',        'small shark',         'mako shark',
           '1m shark',     'blacktip shark',           'ft shark',
          '6ft shark',        'large shark',      'sandbar shark',
              'shark',  'raggedtooth shark',    'sevengill shark',
     'whitetip shark',    'galapagos shark',         'blue shark',
        'wfite shark',    'wobbegong shark',        'rreef shark',
         'horn shark',       'whaler shark',           '6m shark',
       'tiger  shark',    'sandtiger shark',        'while shark',
    'epaulette shark',      'spinner shark',           'no shark',
         'tope shark',     'juvenile shark',           '4m shark',
            'a shark',            'm shark'

In [252]:
unknown_shark_name_df = shark_attack_df[
    shark_attack_df["shark_name"] == "unknown"
]

unknown_activity_df.shape

(296, 12)

In [253]:
shark_attack_df["shark_name"] = (
    shark_attack_df["shark_name"]
    .fillna("unknown")
)

In [254]:
shark_attack_df.to_excel("shark_attack_clean.xlsx", index=False)